> **Outputs cleared 2026-08-15.** These notebooks are exploratory views onto the
> pipeline, not a source of paper numbers. Their stored outputs came from the
> pre-audit run, so they were cleared rather than left to look authoritative.
> Run the cells yourself against the current `results/` CSVs; the numbers quoted
> in the manuscript come from `paper/notes/REWRITE_LEDGER.md` and the result
> tables it cites, never from here.


# 02 — The baseline ladder and the Kalman result

Question: after removing trend + seasonality (fit on training data only, per fold), is there
any predictability left beyond persistence-style baselines?

Answer, in three steps:
1. Damped persistence beats naive persistence and climatology everywhere.
2. Per-basin ridge on 5 own lags beats damped persistence at 1–4 months (+2–5%, DM p<1e-4).
3. A 3-parameter Kalman filter (AR(1) state + observation noise) beats the ridge at **every**
   lead — most of the "skill above persistence" is optimal filtering of GRACE noise.

Results are produced by `scripts/run_phase2_baselines.py` and `scripts/run_kalman_baseline.py`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from gracefc.models import rmse

summary = pd.read_csv(ROOT / "results/phase2_baseline_summary.csv")
summary.head()

In [ ]:
# Ladder in standardized units: lower is better; damped persistence is the reference
order = ["climatology_zero", "persistence", "damped_persistence_reg",
         "ridge_own_lags", "ridge_own_perbasin"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for m in order:
    sub = summary[summary["model"] == m].sort_values("horizon")
    ax.plot(sub["horizon"], sub["rmse_std"], marker="o", label=m)
kal = pd.read_csv(ROOT / "results/kalman_predictions.csv", parse_dates=["target_date"])
kr = kal.groupby("horizon").apply(lambda g: rmse(g["target"].values, g["pred"].values), include_groups=False)
ax.plot(kr.index, kr.values, marker="s", lw=2.5, color="black", label="kalman_ar1")
ax.set_xlabel("lead (months)"); ax.set_ylabel("pooled RMSE (train-std units)")
ax.set_title("Baseline ladder on deseasonalized TWSA, 234 basins")
ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
# Skill of each model vs damped persistence, by lead
piv = summary.pivot(index="model", columns="horizon", values="rmse_std")
damped = piv.loc["damped_persistence_reg"]
skill = (1 - (piv / damped) ** 2) * 100
skill.loc["kalman_ar1"] = (1 - (kr / damped) ** 2) * 100
skill.round(2).sort_values(1, ascending=False)

**Why the Kalman filter wins (errors-in-variables in one sentence):** damped persistence decays
the *observed* current month, noise included; the Kalman filter first estimates how much of the
current observation is signal, then decays that — and with GRACE observation noise at 1–2 cm,
that distinction is worth ~5% at short leads and persists to lead 6.

**Paper implication:** any ML/graph model claiming skill on deseasonalized TWSA must beat this
3-parameter filter, not just persistence — a bar most of the literature has not applied.